In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

# Reload the dataset
transform = transforms.ToTensor()
train_dataset = torchvision.datasets.MNIST(
    root='../data/',
    train=True,
    download=True,
    transform=transform
)

# DataLoader - same as before
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)

print("Setup complete!")
print(f"Total training images: {len(train_dataset)}")

Setup complete!
Total training images: 60000


In [2]:
class ExpertCNN(nn.Module):
    def __init__(self):
        super(ExpertCNN, self).__init__()
        # Convolutions to find features like loops and lines
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)  # Shrinks the image by half each time
        
        # Dense layers to make the final decision (0 through 9)
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 10)  # 10 outputs for 10 possible digits

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))  # Size: 14x14
        x = self.pool(F.relu(self.conv2(x)))  # Size: 7x7
        x = x.view(-1, 64 * 7 * 7)            # Flatten
        x = F.relu(self.fc1(x))
        x = self.fc2(x)                       # Final 10 scores
        return x

print("ExpertCNN model defined!")

ExpertCNN model defined!


In [3]:
# Create the Expert
expert_model = ExpertCNN()

# Multiple-choice loss function for classification
classification_loss = nn.CrossEntropyLoss()

# Optimizer
expert_optimizer = optim.Adam(expert_model.parameters(), lr=1e-3)

print("Expert model initialized!")
print(f"Total parameters: {sum(p.numel() for p in expert_model.parameters()):,}")

Expert model initialized!
Total parameters: 421,642


In [4]:
##training
expert_epochs = 3  # CNNs learn MNIST very quickly

print("Starting to train the Expert CNN on CLEAN data...")
print("-" * 45)

for epoch in range(expert_epochs):
    correct_guesses = 0
    total_images = 0

    for batch_idx, (clean_images, labels) in enumerate(train_loader):

        # 1. Reset gradients
        expert_optimizer.zero_grad()

        # 2. Get the Expert's predictions
        predictions = expert_model(clean_images)

        # 3. Grade the Expert (compare predictions to actual labels)
        loss = classification_loss(predictions, labels)

        # 4. Update the brain
        loss.backward()
        expert_optimizer.step()

        # Track accuracy
        _, predicted_class = torch.max(predictions.data, 1)
        total_images += labels.size(0)
        correct_guesses += (predicted_class == labels).sum().item()

    accuracy = 100 * correct_guesses / total_images
    print(f"Epoch {epoch+1}/3 complete  |  Accuracy: {accuracy:.2f}%")

print("-" * 45)
print("Expert is fully trained!")

Starting to train the Expert CNN on CLEAN data...
---------------------------------------------
Epoch 1/3 complete  |  Accuracy: 92.87%
Epoch 2/3 complete  |  Accuracy: 98.14%
Epoch 3/3 complete  |  Accuracy: 98.73%
---------------------------------------------
Expert is fully trained!


In [5]:
import os

# Create the models folder if it doesn't exist
os.makedirs('./models', exist_ok=True)

# Save the Expert
torch.save(expert_model.state_dict(), './models/expert_cnn.pth')
print("Expert model saved to ./models/expert_cnn.pth")

print("-" * 45)
print("All models saved! You won't need to retrain tomorrow.")

Expert model saved to ./models/expert_cnn.pth
---------------------------------------------
All models saved! You won't need to retrain tomorrow.


In [6]:
# Verify the Expert actually works on some sample images
expert_model.eval()

digit_names = ['zero','one','two','three','four',
               'five','six','seven','eight','nine']

with torch.no_grad():
    test_images = torch.stack([train_dataset[i][0] for i in range(10)])
    test_labels = [train_dataset[i][1] for i in range(10)]
    
    outputs = expert_model(test_images)
    _, predicted = torch.max(outputs, 1)

print("Sample Predictions vs Actual Labels:")
print("-" * 35)
for i in range(10):
    actual    = test_labels[i]
    predicted_label = predicted[i].item()
    status    = "✓" if actual == predicted_label else "✗"
    print(f"  {status}  Actual: {actual}  |  Predicted: {predicted_label}")

Sample Predictions vs Actual Labels:
-----------------------------------
  ✓  Actual: 5  |  Predicted: 5
  ✓  Actual: 0  |  Predicted: 0
  ✓  Actual: 4  |  Predicted: 4
  ✓  Actual: 1  |  Predicted: 1
  ✓  Actual: 9  |  Predicted: 9
  ✓  Actual: 2  |  Predicted: 2
  ✓  Actual: 1  |  Predicted: 1
  ✓  Actual: 3  |  Predicted: 3
  ✓  Actual: 1  |  Predicted: 1
  ✓  Actual: 4  |  Predicted: 4


In [7]:
import os
os.makedirs('../models', exist_ok=True)

torch.save(expert_model.state_dict(), '../models/expert_cnn.pth')
print("Expert saved!", os.path.abspath('../models/expert_cnn.pth'))

Expert saved! C:\Users\Rushikesh\OneDrive\CODES\SelfHealingNN\models\expert_cnn.pth
